In [0]:
#Load Tables
from pyspark.sql.functions import *

telemetry = spark.table(
    "workspace.predictive_maintenance.silver_telemetry"
)

failures = spark.table(
    "workspace.predictive_maintenance.silver_failures"
)

maint = spark.table(
    "workspace.predictive_maintenance.silver_maint"
)

In [0]:
#Failure Count per Machine
failure_count = failures.groupBy(
    "machineID"
).count().withColumnRenamed(
    "count",
    "failure_count"
)

display(failure_count)

machineID,failure_count
1,7
10,5
11,6
13,13
22,15
31,7
33,9
34,3
43,10
45,7


In [0]:
#Maintenance Count per Machine
maintenance_count = maint.groupBy(
    "machineID"
).count().withColumnRenamed(
    "count",
    "maintenance_count"
)

display(maintenance_count)

machineID,maintenance_count
1,37
2,32
3,37
4,33
5,35
6,34
7,37
8,29
9,34
10,38


In [0]:
#Average Telemetry per Machine
telemetry_summary = telemetry.groupBy(
    "machineID"
).agg(
    avg("volt").alias("avg_volt"),
    avg("rotate").alias("avg_rotate"),
    avg("pressure").alias("avg_pressure"),
    avg("vibration").alias("avg_vibration")
)

display(telemetry_summary)

machineID,avg_volt,avg_rotate,avg_pressure,avg_vibration
1,170.83389762467777,446.3365017291676,100.66830588816336,40.5863088734261
2,170.76048201063625,446.38891465251436,100.53992979635984,40.301538634559726
3,170.6592348040541,446.5832379901177,100.65311416397488,40.47537610412395
4,170.6597277050074,446.0538013768685,100.56990183131273,40.34161964770795
5,171.04443470435896,446.8585765042036,101.06618841696498,40.264845827522414
6,170.41171548820324,448.5132691907918,100.55627717638285,40.380021869144485
7,171.1800760927571,446.09459516049526,100.82658081265689,40.4901243964111
8,170.59621853024865,447.02614828161074,100.58832750315484,40.367218838073725
9,170.69206861089256,445.7249744682517,100.77190261281486,40.27650668528428
10,170.64417445035767,446.0354167393137,100.50395234193846,40.27462776712498


In [0]:
#Create Reliability Dataset
reliability = telemetry_summary \
    .join(failure_count, "machineID", "left") \
    .join(maintenance_count, "machineID", "left") \
    .fillna(0)

display(reliability)

machineID,avg_volt,avg_rotate,avg_pressure,avg_vibration,failure_count,maintenance_count
34,170.9048208577788,447.4467304983512,100.51710354500632,40.27782244589459,3,32
37,170.55459909641792,445.53931460904994,101.24704758636787,40.69479044693231,14,35
47,170.9726347097134,445.34503972751355,100.8469650015395,40.214861321961834,10,27
79,170.86596247470044,446.49103985096576,100.64750419405841,40.435518722302604,10,34
24,170.68119558805935,447.01802693933473,101.11024891456462,40.4414669787482,10,38
90,170.9921268232847,445.34179454205616,101.31720147905374,40.28118657723847,12,34
58,170.71075432663156,445.80276585707264,101.20405721584818,40.25956725435997,8,31
12,170.9371764931595,447.2425567224375,100.5721741211863,40.295779585232815,6,31
69,170.43275693741577,446.11516546546954,101.09881777111772,40.50902203216637,10,34
83,170.90485976018903,445.46782046679965,100.81762563445692,40.638414264591006,14,34


In [0]:
#Reliability Score
reliability_score = reliability.withColumn(
    "reliability_score",
    100
    - (col("failure_count") * 10)
    - (col("maintenance_count") * 2)
)

display(reliability_score)

machineID,avg_volt,avg_rotate,avg_pressure,avg_vibration,failure_count,maintenance_count,reliability_score
34,170.9048208577788,447.4467304983512,100.51710354500632,40.27782244589459,3,32,6
37,170.55459909641792,445.53931460904994,101.24704758636787,40.69479044693231,14,35,-110
47,170.9726347097134,445.34503972751355,100.8469650015395,40.214861321961834,10,27,-54
79,170.86596247470044,446.49103985096576,100.64750419405841,40.435518722302604,10,34,-68
24,170.68119558805935,447.01802693933473,101.11024891456462,40.4414669787482,10,38,-76
90,170.9921268232847,445.34179454205616,101.31720147905374,40.28118657723847,12,34,-88
58,170.71075432663156,445.80276585707264,101.20405721584818,40.25956725435997,8,31,-42
12,170.9371764931595,447.2425567224375,100.5721741211863,40.295779585232815,6,31,-22
69,170.43275693741577,446.11516546546954,101.09881777111772,40.50902203216637,10,34,-68
83,170.90485976018903,445.46782046679965,100.81762563445692,40.638414264591006,14,34,-108


In [0]:
#Most Reliable Machines
most_reliable = reliability_score.orderBy(
    col("reliability_score").desc()
)

display(
    most_reliable.limit(10)
)

machineID,avg_volt,avg_rotate,avg_pressure,avg_vibration,failure_count,maintenance_count,reliability_score
77,170.54616238392248,447.5733947820554,100.63570636358268,40.264762229270474,0,33,34
6,170.41171548820324,448.5132691907918,100.55627717638285,40.380021869144485,0,34,32
72,170.67975896159686,448.5645742720318,100.7488963603337,40.25159187038177,2,29,22
100,170.69247100929613,448.2770771542642,100.68062979676372,40.321833091984566,3,28,14
57,170.81728120273817,447.3566879713645,100.58118710752568,40.28978954436252,2,34,12
29,170.79230647553672,447.035142781294,100.67887719131112,40.261928860970464,2,34,12
46,170.74180226946726,448.1057919225283,100.72534144935192,40.25714767938854,2,36,8
34,170.9048208577788,447.4467304983512,100.51710354500632,40.27782244589459,3,32,6
41,170.8183095508191,447.82338681790793,100.77526914276298,40.22854839896316,3,34,2
61,170.64795195660383,446.90936762439117,100.50024616155146,40.32145135424693,4,30,0


In [0]:
least_reliable = reliability_score.orderBy(
    col("reliability_score")
)

display(
    least_reliable.limit(10)
)

machineID,avg_volt,avg_rotate,avg_pressure,avg_vibration,failure_count,maintenance_count,reliability_score
99,170.72788359486455,444.7664879806162,101.56885226044444,40.52472953266788,19,34,-158
98,170.78247826672967,446.06697430626855,101.70549772443894,40.49960830476827,16,37,-134
22,170.72227540045975,445.5194430567094,101.50622780531138,40.51131920660993,15,33,-116
17,170.726236739353,445.96852475599985,101.34290760938606,40.54904458332405,15,32,-114
37,170.55459909641792,445.53931460904994,101.24704758636787,40.69479044693231,14,35,-110
83,170.90485976018903,445.46782046679965,100.81762563445692,40.638414264591006,14,34,-108
20,170.95206335631127,446.56120259853805,101.3084514033588,40.56736476306904,13,33,-96
71,170.90093009515783,446.55202987326624,101.30413812360636,40.48126782558693,13,33,-96
13,170.9086706747842,446.6830123087084,101.25629649945613,40.451197978656154,13,33,-96
90,170.9921268232847,445.34179454205616,101.31720147905374,40.28118657723847,12,34,-88


In [0]:
display(
    reliability_score.select(
        "machineID",
        "reliability_score"
    )
)

machineID,reliability_score
37,-110
27,-6
71,-96
100,14
88,-68
80,-12
94,-72
47,-54
12,-22
46,8


Databricks visualization. Run in Databricks to view.

In [0]:
display(
    failure_count.orderBy(
        col("failure_count").desc()
    )
)

machineID,failure_count
99,19
98,16
17,15
22,15
37,14
83,14
71,13
20,13
13,13
16,13


Databricks visualization. Run in Databricks to view.

In [0]:
display(
    maintenance_count.orderBy(
        col("maintenance_count").desc()
    )
)

machineID,maintenance_count
66,39
70,39
68,39
24,38
27,38
42,38
14,38
84,38
10,38
52,37


Databricks visualization. Run in Databricks to view.